# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [ ]:
from IPython.display import Markdown, display
from openai import OpenAI

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

class TinyScraper:
    def __init__(self, headless=False):
        options = Options()
        if headless:
            options.add_argument("--headless=new")

        self.driver = webdriver.Chrome(options=options)

    def scrape(self, url):
        self.driver.get(url)

        title = self.driver.title

        # Recursive DOM text extraction via JS
        content = self.driver.execute_script("""
            function getText(node) {
                let text = "";

                // Skip script/style/noscript
                if (node.nodeName === "SCRIPT" || 
                    node.nodeName === "STYLE" || 
                    node.nodeName === "NOSCRIPT") {
                    return "";
                }

                // If text node
                if (node.nodeType === Node.TEXT_NODE) {
                    return node.nodeValue.trim() + " ";
                }

                // Recurse children
                for (let child of node.childNodes) {
                    text += getText(child);
                }

                return text;
            }

            return getText(document.body);
        """)

        return {
            "url": url,
            "title": title,
            "content": content[:2000]
        }

    def close(self):
        self.driver.quit()



OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."


def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website['title']}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website['text']
    return user_prompt


def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

def summarize(url):
    scraper = TinyScraper()
    data = scraper.scrape("https://openai.com")
    scraper.close()
    website = {
        "url": data["url"],
        "title": data["title"],
        "text": data["content"]
    }    
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://openai.com")